cd contest2
source venv/bin/activate
cd ../
export LD_LIBRARY_PATH=$VIRTUAL_ENV/lib/python3.12/site-packages/nvidia/cudnn/lib:$VIRTUAL_ENV/lib/python3.12/site-packages/nvidia/cublas/lib:$VIRTUAL_ENV/lib/python3.12/site-packages/nvidia/cusolver/lib:$VIRTUAL_ENV/lib/python3.10/site-packages/nvidia/cusparse/lib
python3 -c "import tensorflow as tf; print('Доступные GPU:', tf.config.list_physical_devices('GPU'))"
jupyter notebook --no-browser

In [1]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB2

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_DIR = Path('.')
TRAIN_DIR = DATA_DIR / 'train'
TEST_DIR = DATA_DIR / 'test'
LABELS_PATH = DATA_DIR / 'labels.csv'
SAMPLE_SUBMISSION_PATH = DATA_DIR / 'sample_submission.csv'

IMG_SIZE = 256
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

I0000 00:00:1779887258.343417    5340 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("включено")
    except RuntimeError as e:
        print(e)

включено


In [3]:
labels = pd.read_csv(LABELS_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

labels['filename'] = labels['id'].astype(str) + '.jpg'
labels['filepath'] = labels['filename'].apply(lambda name: str(TRAIN_DIR / name))

classes = sorted(labels['breed'].unique())
class_to_index = {breed: idx for idx, breed in enumerate(classes)}
labels['label'] = labels['breed'].map(class_to_index)

train_df, valid_df = train_test_split(
    labels,
    test_size=0.2,
    random_state=SEED,
    stratify=labels['breed'],
)


In [4]:
def decode_image(path, label=None):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    if label is None:
        return image
    return image, tf.one_hot(label, depth=len(classes))

def make_dataset(df, training=False):
    paths = df['filepath'].values
    labels_array = df['label'].values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels_array))
    if training:
        ds = ds.shuffle(buffer_size=len(df), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, training=True)
valid_ds = make_dataset(valid_df, training=False)

I0000 00:00:1779887286.242939    5340 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3582 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


In [5]:
augmentation = keras.Sequential(
    [
        layers.RandomFlip('horizontal', seed=SEED),
        layers.RandomRotation(0.15, seed=SEED),
        layers.RandomZoom(0.15, seed=SEED),
        layers.RandomTranslation(0.1, 0.1, seed=SEED),
        layers.RandomBrightness(factor=0.2, seed=SEED),
        layers.RandomContrast(factor=0.2, seed=SEED),
    ],
    name='augmentation',
)



In [6]:
def build_model():
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = augmentation(inputs)

    base_model = EfficientNetB2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet',
    )
    base_model.trainable = False 

    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x) 
    
    outputs = layers.Dense(
        len(classes), 
        activation='softmax',
        kernel_regularizer=keras.regularizers.l2(1e-4)
    )(x)

    model = keras.Model(inputs, outputs)
    return model, base_model

model, base_model = build_model()

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=5, name='top_5_accuracy')],
)


In [7]:
callbacks_phase1 = [
    keras.callbacks.ModelCheckpoint(
        'efficientnetb2_dog_breeds_phase1.keras',
        monitor='val_loss',
        save_best_only=True,
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=2,
        min_lr=1e-6,
    ),
]

initial_epochs = 6
history_frozen = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=initial_epochs,
    callbacks=callbacks_phase1,
)



Epoch 1/6


I0000 00:00:1779887304.691165    5422 cuda_dnn.cc:461] Loaded cuDNN version 92200


512/512 ━━━━━━━━━━━━━━━━━━━━ 71s 109ms/step - accuracy: 0.5562 - loss: 2.1957 - top_5_accuracy: 0.8047 - val_accuracy: 0.8709 - val_loss: 0.6599 - val_top_5_accuracy: 0.9917 - learning_rate: 0.0010
Epoch 2/6
512/512 ━━━━━━━━━━━━━━━━━━━━ 50s 97ms/step - accuracy: 0.7440 - loss: 1.0326 - top_5_accuracy: 0.9414 - val_accuracy: 0.8841 - val_loss: 0.5015 - val_top_5_accuracy: 0.9922 - learning_rate: 0.0010
Epoch 3/6
512/512 ━━━━━━━━━━━━━━━━━━━━ 53s 104ms/step - accuracy: 0.7730 - loss: 0.8891 - top_5_accuracy: 0.9559 - val_accuracy: 0.8870 - val_loss: 0.4736 - val_top_5_accuracy: 0.9936 - learning_rate: 0.0010
Epoch 4/6
512/512 ━━━━━━━━━━━━━━━━━━━━ 50s 98ms/step - accuracy: 0.7927 - loss: 0.8223 - top_5_accuracy: 0.9634 - val_accuracy: 0.8895 - val_loss: 0.4728 - val_top_5_accuracy: 0.9922 - learning_rate: 0.0010
Epoch 5/6
512/512 ━━━━━━━━━━━━━━━━━━━━ 52s 102ms/step - accuracy: 0.8108 - loss: 0.7659 - top_5_accuracy: 0.9686 - val_accuracy: 0.8885 - val_loss: 0.4746 - val_top_5_accuracy: 0.9

In [8]:

base_model.trainable = True
fine_tune_at = len(base_model.layers) - 40
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False
for layer in base_model.layers[fine_tune_at:]:
    layer.trainable = True

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=5, name='top_5_accuracy')],
)

callbacks_phase2 = [
    keras.callbacks.ModelCheckpoint(
        'efficientnetb2_dog_breeds_best.keras',
        monitor='val_loss',
        save_best_only=True,
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

fine_tune_epochs = 10
total_epochs = initial_epochs + fine_tune_epochs

history_fine = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=total_epochs,
    initial_epoch=history_frozen.epoch[-1] + 1,
    callbacks=callbacks_phase2,
)


Epoch 7/16


E0000 00:00:1779887630.144863    5340 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_1_1/efficientnetb2_1/block1b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


512/512 ━━━━━━━━━━━━━━━━━━━━ 83s 132ms/step - accuracy: 0.6778 - loss: 1.3976 - top_5_accuracy: 0.9162 - val_accuracy: 0.8636 - val_loss: 0.5821 - val_top_5_accuracy: 0.9907 - learning_rate: 1.0000e-05
Epoch 8/16
512/512 ━━━━━━━━━━━━━━━━━━━━ 65s 126ms/step - accuracy: 0.7175 - loss: 1.1907 - top_5_accuracy: 0.9307 - val_accuracy: 0.8685 - val_loss: 0.5436 - val_top_5_accuracy: 0.9922 - learning_rate: 1.0000e-05
Epoch 9/16
512/512 ━━━━━━━━━━━━━━━━━━━━ 66s 129ms/step - accuracy: 0.7347 - loss: 1.0912 - top_5_accuracy: 0.9435 - val_accuracy: 0.8733 - val_loss: 0.5233 - val_top_5_accuracy: 0.9922 - learning_rate: 1.0000e-05
Epoch 10/16
512/512 ━━━━━━━━━━━━━━━━━━━━ 64s 125ms/step - accuracy: 0.7442 - loss: 1.0207 - top_5_accuracy: 0.9511 - val_accuracy: 0.8758 - val_loss: 0.5074 - val_top_5_accuracy: 0.9936 - learning_rate: 1.0000e-05
Epoch 11/16
512/512 ━━━━━━━━━━━━━━━━━━━━ 64s 125ms/step - accuracy: 0.7630 - loss: 0.9708 - top_5_accuracy: 0.9529 - val_accuracy: 0.8768 - val_loss: 0.4985 -

In [9]:
test_paths = [str(TEST_DIR / f'{image_id}.jpg') for image_id in sample_submission['id']]
test_ds = tf.data.Dataset.from_tensor_slices(test_paths)
test_ds = test_ds.map(lambda path: decode_image(path), num_parallel_calls=AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

predictions = model.predict(test_ds, verbose=0)

submission = pd.DataFrame(predictions, columns=classes)
submission.insert(0, 'id', sample_submission['id'].values)
submission = submission[sample_submission.columns]
submission.to_csv('submission3t.csv', index=False)
